# EKF comparison · constant and variable CO₂ solubility

Each figure compares **one chemical model** under the two existing gas formulations. S and G/F are shown separately; only the two EKF trajectories appear in each panel.

- Same chemical parameters, CO₂ observations, causal 30-minute windows and filter noise settings; offline chemistry is used for evaluation only.
- Constant formulation: $C_{sat}=1.5$ g/L, $k_La=0.074679$ h⁻¹, gain $=0.635239$.
- Variable formulation: $C_{sat}=3.295\exp[(7\times10^{-5}a-0.02905)T-0.0179a-0.00111S]p_{CO_2}$, where $a=E/7.8924$ (% vol), $T$ is °C, $S$ is g/L and $p_{CO_2}=1$ bar. Recalibrated $(k_La, gain)$: S $(0.137966,0.658678)$; G/F $(0.150080,0.646064)$.

This compares the **complete current formulations**, including the revised gas transfer/desorption law and recalibrated gas parameters. It does not isolate the solubility correlation alone. Gas calibration: LAB019–021; gas holdout: LAB016–018. Chemical calibration used separate fermentations; the analytical comparisons here are holdout evaluations.

RMSE has the plotted variable’s units. YAN includes nutrition samples. Initial values remain visible as Samples and are excluded from post-initial chemical RMSE; substituted zero G+F values retain the existing censoring policy. No chemical sample is assimilated.

The solubility correlation is reproduced from [Liger-Belair et al. (2023), Eq. 1](https://doi.org/10.1021/acsomega.3c01812). Its use for fermenting must and the assumed CO₂ partial pressure remain modeling assumptions.

In [ ]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nbformat
from IPython.display import display
from IPython.utils.capture import capture_output
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/wine_fermentation').is_dir())
INPUTS = {
    'Constant solubility': ROOT/'notebooks/EKF/03_zenteno_single_sugar_co2_only.source.ipynb',
    'Variable solubility': ROOT/'notebooks/EKF/06_zenteno_single_sugar_variable_solubility.source.ipynb',
}
INPUT_HASHES = {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in INPUTS.values()}
def run_filters(label):
    notebook = nbformat.read(INPUTS[label], as_version=4)
    namespace = {'__name__': '__main__'}
    # Recompute the original data preparation and both EKFs, without running
    # their mixed-model plotting/evaluation cell or modifying any input file.
    with capture_output() as log:
        for index in (1, 3, 5):
            exec(compile(notebook.cells[index].source, str(INPUTS[label])+f':cell{index}', 'exec'), namespace)
    print(label + ': six fermentations completed for both models.')
    return namespace
RUNS = {}


In [ ]:
RUNS['Constant solubility'] = run_filters('Constant solubility')

In [ ]:
RUNS['Variable solubility'] = run_filters('Variable solubility')

In [ ]:
LABS = RUNS['Variable solubility']['LABS']
TRAIN = RUNS['Variable solubility']['TRAIN']
REFERENCE = RUNS['Variable solubility']
v = REFERENCE['v']
chemical_observations = REFERENCE['chemical_observations']
CHEMICAL_COLUMNS = REFERENCE['CHEMICAL_COLUMNS']
COLORS = {'Constant solubility': '#D55E00', 'Variable solubility': '#0072B2'}
STYLES = {'Constant solubility': '--', 'Variable solubility': '-'}
RESULTS = {model: {label: ns['RESULTS' if model == 'S' else 'RESULTS_GF']
                   for label, ns in RUNS.items()} for model in ('S', 'G/F')}
for model, cases in RESULTS.items():
    for lab in LABS:
        a, b = [res[lab]['sensor'] for res in cases.values()]
        for col in ('t_h', 'measured', 'valid', 'assimilate'):
            np.testing.assert_allclose(a[col], b[col], equal_nan=True)
        for state in ('X', 'N', 'GF', 'E', 'Cd'):
            np.testing.assert_allclose(cases['Constant solubility'][lab]['frame']['EKF_'+state].iloc[0],
                                       cases['Variable solubility'][lab]['frame']['EKF_'+state].iloc[0])
METRICS = []
FIGURES = {}
def observation_arrays(lab, state):
    if lab in TRAIN:
        o = chemical_observations(v.chem, lab, state)
        tt = o.t_h.to_numpy(); yy = o[CHEMICAL_COLUMNS[state]].to_numpy()
    elif state == 'X':
        o = v.biomass.loc[v.biomass.lab.eq(lab)].dropna(subset=['t_h', 'X_viable_g_l'])
        tt = o.t_h.to_numpy(); yy = o.X_viable_g_l.to_numpy()
    else:
        o = pd.DataFrame(); tt = np.array([]); yy = np.array([])
    mask = np.isfinite(tt) & np.isfinite(yy) & (tt > 0) & (tt <= v.horizons[lab])
    if state == 'GF' and 'GF_zero_substitution' in o:
        mask &= ~o.GF_zero_substitution.astype(str).str.lower().eq('true').to_numpy()
    return o, tt, yy, mask

def compare(model, state, title):
    factor = 1000 if state == 'N' else 1
    unit = 'g/(L h)' if state == 'CO2' else ('mg/L' if state == 'N' else 'g/L')
    fig, axs = plt.subplots(2, 3, figsize=(15, 7.4))
    for ax, lab in zip(axs.flat, LABS):
        errors = {}
        if state == 'CO2':
            sensor = RESULTS[model]['Variable solubility'][lab]['sensor']
            ax.plot(sensor.t_h, sensor.measured, c='black', lw=.7, label='Samples')
        elif state != 'Cd':
            o, tt, yy, mask = observation_arrays(lab, state)
            sample_label = 'G+F' if state == 'GF' else 'Samples'
            ax.scatter(tt[mask], yy[mask]*factor, c='black', s=20, label=sample_label, zorder=5)
            first = RESULTS[model]['Variable solubility'][lab]['frame']
            ax.scatter([0], [first['EKF_'+state].iloc[0]*factor], c='black', s=20, label=sample_label, zorder=5)
            if state == 'GF':
                density = v.density.loc[v.density.lab.eq(lab)].dropna(subset=['t_h','density_g_l'])
                density = density.loc[density.t_h.between(0, v.horizons[lab])]
                ax.scatter(density.t_h, 2.5*(density.density_g_l-1000), marker='D', facecolors='none',
                           edgecolors='#009E73', s=22, label='G+F from density', zorder=4)
        for label, results in RESULTS[model].items():
            result = results[lab]
            if state == 'CO2':
                q = result['sensor']; tx = q.t_h; prediction = q.forecast
                valid = q.valid & np.isfinite(q.measured)
                error = q.loc[valid, 'forecast'] - q.loc[valid, 'measured']
            else:
                f = result['frame']; tx = f.t_h; prediction = f['EKF_'+state]*factor
                error = (np.interp(tt[mask], f.t_h, f['EKF_'+state])-yy[mask])*factor if state != 'Cd' else []
            ax.plot(tx, prediction, color=COLORS[label], ls=STYLES[label], lw=1.5, label=label)
            if len(error):
                rmse = float(np.sqrt(np.mean(np.asarray(error)**2))); errors[label] = rmse
                METRICS.append(dict(model=model, solubility=label, LAB=lab, variable=state,
                                    n=len(error), RMSE=rmse, unit=unit,
                                    gas_split='Calibration' if lab in TRAIN else 'Holdout'))
        for t, dose in REFERENCE['batches'][lab].pulses['N']:
            ax.axvline(t, c='gray', ls=':', lw=.8, label='N pulse')
        handles, labels = ax.get_legend_handles_labels()
        unique = dict(zip(labels, handles))
        legend_title = '\n'.join(f'RMSE {label} = {value:.3f} {unit}' for label, value in errors.items())
        ax.legend(unique.values(), unique.keys(), loc='best', fontsize=6.8, title=legend_title or None,
                  title_fontsize=6.8, framealpha=.9)
        ax.set(title=lab, xlabel='Time [h]', ylabel=title+' ['+unit+']')
        ax.grid(alpha=.2)
    context = 'forecast before correction' if state == 'CO2' else ('model state' if state == 'Cd' else 'chemical holdout')
    fig.suptitle(f'{title} · EKF {model} · {context}', y=.995)
    fig.tight_layout(pad=.65, h_pad=1.0, w_pad=.8, rect=(0, 0, 1, .975))
    FIGURES[(model, state)] = fig
    plt.show()


## CO₂

In [ ]:
compare('S', 'CO2', 'CO₂')

In [ ]:
compare('G/F', 'CO2', 'CO₂')

## Biomass

In [ ]:
compare('S', 'X', 'Biomass')

In [ ]:
compare('G/F', 'X', 'Biomass')

## YAN

In [ ]:
compare('S', 'N', 'YAN')

In [ ]:
compare('G/F', 'N', 'YAN')

## Glucose + fructose

In [ ]:
compare('S', 'GF', 'Glucose + fructose')

In [ ]:
compare('G/F', 'GF', 'Glucose + fructose')

## Ethanol

In [ ]:
compare('S', 'E', 'Ethanol')

In [ ]:
compare('G/F', 'E', 'Ethanol')

## Dissolved CO₂

In [ ]:
compare('S', 'Cd', 'Dissolved CO₂')

In [ ]:
compare('G/F', 'Cd', 'Dissolved CO₂')

## Individual sugars · G/F model

In [ ]:
compare('G/F', 'G', 'Glucose')
compare('G/F', 'F', 'Fructose')

## RMSE comparison

CO₂ rows distinguish gas calibration and holdout. Chemical rows compare independent analytical samples. The summary is the arithmetic mean of per-fermentation RMSE, not a pooled RMSE. Negative change means a lower RMSE for variable solubility. No RMSE is reported for dissolved CO₂ without measurements.

In [ ]:
metrics = pd.DataFrame(METRICS)
yan = metrics.loc[metrics.variable.eq('N')]
assert yan.n.eq(12).all() and len(yan) == 12
comparison = metrics.pivot(index=['model','variable','unit','gas_split','LAB','n'], columns='solubility', values='RMSE')
comparison['Change [%]'] = 100*(comparison['Variable solubility']/comparison['Constant solubility']-1)
display(comparison.round(4))
summary_data = metrics.copy()
summary_data['evaluation'] = np.where(summary_data.variable.eq('CO2'), 'Gas '+summary_data.gas_split.str.lower(), 'Chemical holdout')
summary = summary_data.groupby(['model','variable','unit','evaluation','solubility']).RMSE.mean().unstack('solubility')
summary['Change [%]'] = 100*(summary['Variable solubility']/summary['Constant solubility']-1)
display(summary.round(4))
assert all(hashlib.sha256((ROOT/path).read_bytes()).hexdigest() == digest for path, digest in INPUT_HASHES.items())
print('Identical observation windows, masks and initial conditions verified. All 12 post-initial YAN samples per LAB included, including nutrition samples.')
print('Input notebook SHA-256:', json.dumps(INPUT_HASHES, indent=2))


## Interpretation limits

The comparison retains the current filter settings and does not retune their noise matrices. LAB016–018 lack post-initial analytical YAN measurements, so their YAN trajectories have no empirical RMSE. The late YAN rise in the variable-solubility S EKF for LAB017 is a measurement-correction artifact under these settings and cannot be validated from YAN samples. Dissolved CO₂ trajectories are inferred states, not measured validation data.